# PHASE 5 — Blanket Augmentation (Ablation Control, No Evidence-Filtering)

## Cell 1 — Install dependencies

In [1]:
import subprocess, sys
def _pip_install(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
for pkg in ["ultralytics", "opencv-python-headless", "openpyxl"]:
    _pip_install(pkg)
import torch
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

Torch: 2.10.0+cu128 | CUDA available: True
Using device: 0


## Cell 2 — Imports

In [2]:
import os, glob, json, random, shutil, time
import numpy as np
import pandas as pd
import cv2
import yaml as pyyaml
from ultralytics import YOLO

print("Imports OK")

Imports OK


## Cell 3 — Configuration

Blanket = ALL three Phase 3 degradation types applied with EQUAL, unfiltered
probability -- no XAI evidence gate. Everything else (architecture, epochs,
optimizer, baseline augmentation) is identical to Phase 5 Variant 1 for a fair
comparison.

In [3]:
SEED = 42
IMG_SIZE = 640
EVAL_CONF = 0.001
IOU_THRESHOLD = 0.70

TRAIN_EPOCHS = 150
TRAIN_PATIENCE = 30
TRAIN_BATCH = 12
TRAIN_OPTIMIZER = "SGD"
TRAIN_LR0 = 0.01
TRAIN_LRF = 0.01
TRAIN_MOMENTUM = 0.937
TRAIN_WEIGHT_DECAY = 0.0005
TRAIN_WARMUP_EPOCHS = 3.0
TRAIN_WARMUP_MOMENTUM = 0.8
TRAIN_WARMUP_BIAS_LR = 0.1
TRAIN_SEED = 42
TRAIN_DETERMINISTIC = True
TRAIN_CLOSE_MOSAIC = 20
TRAIN_COS_LR = False

BASELINE_AUG = dict(
    hsv_h=0.02, hsv_s=0.7, hsv_v=0.5,
    degrees=10.0, translate=0.1, scale=0.5, shear=0.0, perspective=0.0,
    flipud=0.0, fliplr=0.5, bgr=0.0,
    mosaic=1.0, mixup=0.0, cutmix=0.0, copy_paste=0.0, copy_paste_mode="flip",
    auto_augment="randaugment", erasing=0.4,
)

# ---------------------------------------------------------------------------
# BLANKET augmentation -- all three Phase 3 degradation types, same probability
# and range as Variant 1's Low-Light/Noise (for fair comparison), PLUS Motion
# Blur added (which Variant 1 deliberately excluded due to lack of evidence).
# ---------------------------------------------------------------------------
BLANKET_LOWLIGHT_PROB = 0.35
BLANKET_LOWLIGHT_FACTOR_RANGE = (0.15, 0.85)
BLANKET_NOISE_PROB = 0.35
BLANKET_NOISE_STD_RANGE = (0.02, 0.16)
BLANKET_BLUR_PROB = 0.35
BLANKET_BLUR_KERNEL_RANGE = (5, 17)   # matches Phase 3's motion-blur severity kernels

random.seed(SEED)
np.random.seed(SEED)

OUTPUT_ROOT = "/kaggle/working/phase5_blanket"
TRAINING_DIR = os.path.join(OUTPUT_ROOT, "training")
MODELS_DIR = os.path.join(OUTPUT_ROOT, "models")
RESULTS_DIR = os.path.join(OUTPUT_ROOT, "results")
LOGS_DIR = os.path.join(OUTPUT_ROOT, "logs")
for d in [OUTPUT_ROOT, TRAINING_DIR, MODELS_DIR, RESULTS_DIR, LOGS_DIR]:
    os.makedirs(d, exist_ok=True)

print("Blanket augmentation configuration loaded.")
print("Low-light:", BLANKET_LOWLIGHT_PROB, BLANKET_LOWLIGHT_FACTOR_RANGE)
print("Noise    :", BLANKET_NOISE_PROB, BLANKET_NOISE_STD_RANGE)
print("Blur     :", BLANKET_BLUR_PROB, BLANKET_BLUR_KERNEL_RANGE)

Blanket augmentation configuration loaded.
Low-light: 0.35 (0.15, 0.85)
Noise    : 0.35 (0.02, 0.16)
Blur     : 0.35 (5, 17)


## Cell 4 — Locate Phase 3 output (before-model, clean+degraded test sets)

In [4]:
def find_phase3_root():
    candidates = []
    for root in ("/kaggle/input", "/kaggle/working"):
        for config_path in glob.glob(os.path.join(root, "**", "phase3_config.json"), recursive=True):
            d = os.path.dirname(config_path)
            if os.path.isdir(os.path.join(d, "degraded_datasets")):
                candidates.append(d)
    return sorted(set(candidates))

phase3_candidates = find_phase3_root()
if not phase3_candidates:
    raise FileNotFoundError("No Phase 3 output found. Add the Phase 3 output zip as a Kaggle Input dataset.")
PHASE3_ROOT = phase3_candidates[0]
print("PHASE3_ROOT:", PHASE3_ROOT)

with open(os.path.join(PHASE3_ROOT, "phase3_config.json")) as f:
    phase3_config = json.load(f)

BEFORE_MODEL_PATH = os.path.join(PHASE3_ROOT, "model", "best.pt")
if not os.path.exists(BEFORE_MODEL_PATH):
    cands = glob.glob("/kaggle/input/**/best.pt", recursive=True)
    if not cands:
        raise FileNotFoundError("No baseline best.pt found.")
    BEFORE_MODEL_PATH = cands[0]
print("BEFORE_MODEL_PATH:", BEFORE_MODEL_PATH)

CLEAN_TEST_IMAGES_DIR = os.path.join(PHASE3_ROOT, "clean_test_set", "images")
DEGRADED_DIR = os.path.join(PHASE3_ROOT, "degraded_datasets")
DEGRADATION_DIRS = {
    "Gaussian Noise": os.path.join(DEGRADED_DIR, "gaussian_noise"),
    "Motion Blur": os.path.join(DEGRADED_DIR, "motion_blur"),
    "Low Light": os.path.join(DEGRADED_DIR, "low_light"),
}
for name, d in DEGRADATION_DIRS.items():
    print(f"{name} exists: {os.path.isdir(d)} -> {d}")

SEVERITY_PARAMS = phase3_config.get("severity_params", {})
SEVERITIES = sorted({int(s) for grp in SEVERITY_PARAMS.values() for s in grp.keys()}) or [1, 2, 3, 4]
CLASS_NAMES = phase3_config.get("class_names", ["Fire", "Smoke"])
NUM_CLASSES = len(CLASS_NAMES)
print("Severities:", SEVERITIES, "| Classes:", CLASS_NAMES)

PHASE3_ROOT: /kaggle/input/datasets/sadiahaiderjaima/dataset-p3-using-p5
BEFORE_MODEL_PATH: /kaggle/input/datasets/sadiahaiderjaima/dataset-p3-using-p5/model/best.pt
Gaussian Noise exists: True -> /kaggle/input/datasets/sadiahaiderjaima/dataset-p3-using-p5/degraded_datasets/gaussian_noise
Motion Blur exists: True -> /kaggle/input/datasets/sadiahaiderjaima/dataset-p3-using-p5/degraded_datasets/motion_blur
Low Light exists: True -> /kaggle/input/datasets/sadiahaiderjaima/dataset-p3-using-p5/degraded_datasets/low_light
Severities: [1, 2, 3, 4] | Classes: ['Fire', 'Smoke']


## Cell 5 — Locate / download the training dataset (train+val)

In [5]:
train_data_candidates = glob.glob("/kaggle/input/**/train/images", recursive=True) + \
                         glob.glob("/kaggle/working/**/train/images", recursive=True)

if train_data_candidates:
    TRAIN_DATASET_ROOT = os.path.dirname(os.path.dirname(train_data_candidates[0]))
    print("Using existing dataset at:", TRAIN_DATASET_ROOT)
else:
    print("Downloading from Roboflow...")
    _pip_install("roboflow")
    from roboflow import Roboflow
    ROBOFLOW_API_KEY = "qNZt21Yr4LTdIaH6fUXC"
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace("md-s-workspace").project("fire_smoke_detection-tmqte")
    dataset = project.version(4).download("yolov11")
    TRAIN_DATASET_ROOT = dataset.location
    print("Downloaded to:", TRAIN_DATASET_ROOT)

DATA_YAML_PATH = os.path.join(TRAIN_DATASET_ROOT, "data.yaml")
print("data.yaml:", DATA_YAML_PATH, "| exists:", os.path.exists(DATA_YAML_PATH))

Using existing dataset at: /kaggle/working/fire_smoke_detection-4
data.yaml: /kaggle/working/fire_smoke_detection-4/data.yaml | exists: True


## Cell 6 — Patch build_transforms with BLANKET (unfiltered) augmentation

Idempotent patch (safe to re-run), identical mechanism to Variant 1's patch --
only the transform class differs (adds Motion Blur, no evidence-gate applied).

In [6]:
from ultralytics.data.dataset import YOLODataset

class BlanketDegradationAugment:
    """Blanket (non-evidence-based) augmentation: applies ALL THREE Phase 3
    degradation types with equal, unfiltered probability -- used only as an
    ablation control against the evidence-constrained targeted (Variant 1) run."""
    def __init__(self, p_lowlight, lowlight_range, p_noise, noise_std_range,
                 p_blur, blur_kernel_range):
        self.p_lowlight = p_lowlight
        self.lowlight_range = lowlight_range
        self.p_noise = p_noise
        self.noise_std_range = noise_std_range
        self.p_blur = p_blur
        self.blur_kernel_range = blur_kernel_range

    def _apply_lowlight(self, img):
        factor = random.uniform(*self.lowlight_range)
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32)
        hsv[:, :, 2] = np.clip(hsv[:, :, 2] * factor, 0, 255)
        return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)

    def _apply_noise(self, img):
        std = random.uniform(*self.noise_std_range)
        img_f = img.astype(np.float32) / 255.0
        noise = np.random.normal(0.0, std, img_f.shape).astype(np.float32)
        return (np.clip(img_f + noise, 0, 1) * 255.0).astype(np.uint8)

    def _apply_blur(self, img):
        k = random.choice(range(self.blur_kernel_range[0], self.blur_kernel_range[1] + 1, 2))
        kernel = np.zeros((k, k), dtype=np.float32)
        kernel[k // 2, :] = 1.0 / k
        return cv2.filter2D(img, -1, kernel)

    def __call__(self, labels):
        img = labels["img"]
        if random.random() < self.p_lowlight:
            img = self._apply_lowlight(img)
        if random.random() < self.p_noise:
            img = self._apply_noise(img)
        if random.random() < self.p_blur:
            img = self._apply_blur(img)
        labels["img"] = img
        return labels


if not hasattr(YOLODataset, "_phase5_true_original_build_transforms"):
    current = YOLODataset.__dict__.get("build_transforms")
    while current is not None and getattr(current, "_is_phase5_patch", False):
        current = current._wrapped_original
    YOLODataset._phase5_true_original_build_transforms = current
    print("Captured the TRUE original build_transforms.")
else:
    print("TRUE original build_transforms already captured -- reusing it.")

_true_original = YOLODataset._phase5_true_original_build_transforms

def _patched_build_transforms(self, hyp=None):
    transforms = _true_original(self, hyp)
    if getattr(self, "augment", False):
        blanket = BlanketDegradationAugment(
            p_lowlight=BLANKET_LOWLIGHT_PROB, lowlight_range=BLANKET_LOWLIGHT_FACTOR_RANGE,
            p_noise=BLANKET_NOISE_PROB, noise_std_range=BLANKET_NOISE_STD_RANGE,
            p_blur=BLANKET_BLUR_PROB, blur_kernel_range=BLANKET_BLUR_KERNEL_RANGE,
        )
        insert_at = len(transforms.transforms) - 1 if len(transforms.transforms) > 0 else 0
        transforms.transforms.insert(insert_at, blanket)
        print(f"[Blanket] Inserted BlanketDegradationAugment at position {insert_at} "
              f"(pipeline now has {len(transforms.transforms)} transforms).")
    return transforms

_patched_build_transforms._is_phase5_patch = True
_patched_build_transforms._wrapped_original = _true_original
YOLODataset.build_transforms = _patched_build_transforms
print("Patched YOLODataset.build_transforms (blanket, idempotent).")

Captured the TRUE original build_transforms.
Patched YOLODataset.build_transforms (blanket, idempotent).


## Cell 7 — Evaluation utility (per-class AP50 included)

In [7]:
def make_data_yaml(images_dir, yaml_path, names):
    split_root = os.path.dirname(images_dir)
    data = {"path": split_root, "train": "images", "val": "images",
            "names": {i: n for i, n in enumerate(names)}}
    with open(yaml_path, "w") as f:
        pyyaml.dump(data, f)
    return yaml_path

def evaluate_model(model, images_dir, name, conf=EVAL_CONF, iou=IOU_THRESHOLD, imgsz=IMG_SIZE):
    yaml_path = os.path.join(LOGS_DIR, f"data_{name}.yaml")
    make_data_yaml(images_dir, yaml_path, CLASS_NAMES)
    metrics = model.val(data=yaml_path, imgsz=imgsz, conf=conf, iou=iou, split="val",
                         device=DEVICE, verbose=False, plots=False, save_json=False)
    per_class_ap50 = {}
    try:
        ap50_per_class = metrics.box.ap50
        for i, cname in enumerate(CLASS_NAMES):
            per_class_ap50[f"{cname}_AP50"] = float(ap50_per_class[i]) if i < len(ap50_per_class) else None
    except Exception:
        for cname in CLASS_NAMES:
            per_class_ap50[f"{cname}_AP50"] = None
    return {
        "Precision": float(metrics.box.mp),
        "Recall": float(metrics.box.mr),
        "mAP50": float(metrics.box.map50),
        "mAP50_95": float(metrics.box.map),
        **per_class_ap50,
    }

def evaluate_all_conditions(model, tag):
    rows = []
    m = evaluate_model(model, CLEAN_TEST_IMAGES_DIR, f"{tag}_clean")
    rows.append({"Model": tag, "Degradation": "Clean", "Severity": 0, **m})
    for deg_name, deg_dir in DEGRADATION_DIRS.items():
        deg_key = deg_name.lower().replace(" ", "_")
        for sev in SEVERITIES:
            sev_dir = os.path.join(deg_dir, f"severity_{sev}", "images")
            if not os.path.isdir(sev_dir):
                continue
            m = evaluate_model(model, sev_dir, f"{tag}_{deg_key}_sev{sev}")
            rows.append({"Model": tag, "Degradation": deg_name, "Severity": sev, **m})
    return pd.DataFrame(rows)

print("Evaluation utility ready.")

Evaluation utility ready.


## Cell 8 — BEFORE evaluation (baseline model, for reference)

In [8]:
before_model = YOLO(BEFORE_MODEL_PATH)
before_df = evaluate_all_conditions(before_model, "Before")
before_csv = os.path.join(RESULTS_DIR, "before_results.csv")
before_df.to_csv(before_csv, index=False)
print("Saved:", before_csv)
before_df

Ultralytics 8.4.133 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11s summary (fused): 101 layers, 9,413,574 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 85.7±66.4 MB/s, size: 37.1 KB)
val: Scanning /kaggle/input/datasets/sadiahaiderjaima/dataset-p3-using-p5/clean_test_set/labels... 1715 images, 287 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1715/1715 1.1Kit/s 1.6s0.1s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/sadiahaiderjaima/dataset-p3-using-p5/clean_test_set is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 108/108 5.9it/s 18.4s0.2s
                   all       1715       2790      0.869      0.763      0.828      0.625
Speed: 0.8ms preprocess, 8.1ms inference, 0.0ms loss, 0.6ms postprocess per image
Ultralytics 8.4.133 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
val: Fast image access ✅ (ping: 0.0

,Model,Degradation,Severity,Precision,Recall,mAP50,mAP50_95,Fire_AP50,Smoke_AP50
0,Before,Clean,0,0.869361,0.762934,0.828040,0.625484,0.872152,0.783928
1,Before,Gaussian Noise,1,0.855555,0.716088,0.783998,0.590691,0.851942,0.716054
2,Before,Gaussian Noise,2,0.775929,0.576955,0.636428,0.484954,0.761404,0.511453
3,Before,Gaussian Noise,3,0.783000,0.465214,0.516404,0.402309,0.644998,0.387811
4,Before,Gaussian Noise,4,0.664069,0.356103,0.370867,0.276117,0.513996,0.227737
5,Before,Motion Blur,1,0.877249,0.756887,0.827109,0.624187,0.873843,0.780376
6,Before,Motion Blur,2,0.871974,0.731861,0.809976,0.603373,0.851552,0.768400
7,Before,Motion Blur,3,0.870828,0.690419,0.775272,0.566320,0.818269,0.732274
8,Before,Motion Blur,4,0.817626,0.657045,0.724563,0.514225,0.757586,0.691539
9,Before,Low Light,1,0.875656,0.758979,0.826797,0.627010,0.870845,0.782749


## Cell 9 — Fresh YOLO11s training with BLANKET augmentation

In [ ]:
blanket_model = YOLO("yolo11s.pt")

train_results = blanket_model.train(
    data=DATA_YAML_PATH,
    epochs=TRAIN_EPOCHS,
    patience=TRAIN_PATIENCE,
    batch=TRAIN_BATCH,
    imgsz=IMG_SIZE,
    device=DEVICE,
    project=TRAINING_DIR,
    name="yolo11s_phase5_blanket",
    exist_ok=True,
    optimizer=TRAIN_OPTIMIZER,
    lr0=TRAIN_LR0,
    lrf=TRAIN_LRF,
    momentum=TRAIN_MOMENTUM,
    weight_decay=TRAIN_WEIGHT_DECAY,
    warmup_epochs=TRAIN_WARMUP_EPOCHS,
    warmup_momentum=TRAIN_WARMUP_MOMENTUM,
    warmup_bias_lr=TRAIN_WARMUP_BIAS_LR,
    seed=TRAIN_SEED,
    deterministic=TRAIN_DETERMINISTIC,
    close_mosaic=TRAIN_CLOSE_MOSAIC,
    cos_lr=TRAIN_COS_LR,
    verbose=True,
    plots=False,
    **BASELINE_AUG,
)

BLANKET_MODEL_PATH = os.path.join(TRAINING_DIR, "yolo11s_phase5_blanket", "weights", "best.pt")
print("\nTraining complete.")
print("BLANKET_MODEL_PATH:", BLANKET_MODEL_PATH, "| exists:", os.path.exists(BLANKET_MODEL_PATH))

bundled_blanket_path = os.path.join(MODELS_DIR, "blanket_best.pt")
shutil.copy2(BLANKET_MODEL_PATH, bundled_blanket_path)
print("Copied best.pt to:", bundled_blanket_path)

## Cell 9b — If disconnected mid-training, run this to resume, then re-run Cell 10 onward

In [9]:
last_pt = os.path.join(TRAINING_DIR, "yolo11s_phase5_blanket", "weights", "last.pt")
if os.path.exists(last_pt):
    resumed_model = YOLO(last_pt)
    resumed_model.train(resume=True)
    BLANKET_MODEL_PATH = os.path.join(TRAINING_DIR, "yolo11s_phase5_blanket", "weights", "best.pt")
    bundled_blanket_path = os.path.join(MODELS_DIR, "blanket_best.pt")
    shutil.copy2(BLANKET_MODEL_PATH, bundled_blanket_path)
    print("Resumed and re-bundled:", bundled_blanket_path)
else:
    print("No last.pt found -- nothing to resume (training likely completed normally via Cell 9).")

Ultralytics 8.4.133 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=12, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=20, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/fire_smoke_detection-4/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.02, hsv_s=0.7, hsv_v=0.5, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/working/phase5_blanket/training/yolo11s_phase5_blanket/weights/last

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


     73/150      3.91G      1.008     0.8525      1.259         25        640: 100% ━━━━━━━━━━━━ 674/674 3.9it/s 2:530.3ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 72/72 5.4it/s 13.5s0.2s
                   all       1706       2867      0.856      0.737      0.803      0.574

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     74/150      3.91G     0.9968     0.8443      1.253         28        640: 100% ━━━━━━━━━━━━ 674/674 3.9it/s 2:540.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 72/72 5.3it/s 13.6s0.2s
                   all       1706       2867      0.861      0.734      0.806      0.575

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     75/150      3.91G     0.9998     0.8401      1.252         31        640: 100% ━━━━━━━━━━━━ 674/674 3.8it/s 2:590.2ss
                 Class     Imag

## Cell 10 — AFTER (Blanket) evaluation on the same Clean + degraded sets

In [10]:
blanket_yolo = YOLO(bundled_blanket_path)
blanket_df = evaluate_all_conditions(blanket_yolo, "Blanket")
blanket_csv = os.path.join(RESULTS_DIR, "blanket_results.csv")
blanket_df.to_csv(blanket_csv, index=False)
print("Saved:", blanket_csv)
blanket_df

Ultralytics 8.4.133 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11s summary (fused): 101 layers, 9,413,574 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 62.7±21.8 MB/s, size: 39.8 KB)
val: Scanning /kaggle/input/datasets/sadiahaiderjaima/dataset-p3-using-p5/clean_test_set/labels... 1715 images, 287 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1715/1715 1.1Kit/s 1.6s0.1s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/sadiahaiderjaima/dataset-p3-using-p5/clean_test_set is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 108/108 5.1it/s 21.3s0.2s
                   all       1715       2790      0.867      0.749      0.823      0.617
Speed: 0.8ms preprocess, 9.5ms inference, 0.0ms loss, 0.7ms postprocess per image
Ultralytics 8.4.133 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
val: Fast image access ✅ (ping: 0.7

,Model,Degradation,Severity,Precision,Recall,mAP50,mAP50_95,Fire_AP50,Smoke_AP50
0,Blanket,Clean,0,0.866675,0.748620,0.822813,0.617245,0.867199,0.778428
1,Blanket,Gaussian Noise,1,0.869838,0.748908,0.823093,0.617208,0.869829,0.776358
2,Blanket,Gaussian Noise,2,0.865875,0.740295,0.818875,0.614602,0.865292,0.772459
3,Blanket,Gaussian Noise,3,0.866930,0.732350,0.817555,0.611899,0.867079,0.768031
4,Blanket,Gaussian Noise,4,0.859428,0.725346,0.805946,0.603566,0.860532,0.751359
5,Blanket,Motion Blur,1,0.873934,0.742515,0.821299,0.617391,0.866511,0.776086
6,Blanket,Motion Blur,2,0.861156,0.745431,0.819856,0.616258,0.866465,0.773247
7,Blanket,Motion Blur,3,0.874199,0.733311,0.816586,0.612275,0.863199,0.769972
8,Blanket,Motion Blur,4,0.866732,0.728510,0.811967,0.606107,0.857561,0.766373
9,Blanket,Low Light,1,0.870612,0.745377,0.822780,0.618290,0.868696,0.776865


## Cell 11 — Comparison table: Before vs Blanket (values only, matches Table 2's format)

In [11]:
merged = before_df.merge(blanket_df, on=["Degradation", "Severity"], suffixes=("_Before", "_Blanket"))

for metric in ["Precision", "Recall", "mAP50", "mAP50_95"]:
    merged[f"{metric}_Improvement_Pct"] = (
        (merged[f"{metric}_Blanket"] - merged[f"{metric}_Before"]) / merged[f"{metric}_Before"].replace(0, np.nan)
    ) * 100.0

comparison_csv = os.path.join(RESULTS_DIR, "before_vs_blanket_comparison.csv")
merged.to_csv(comparison_csv, index=False)
print("Saved:", comparison_csv)

cols = ["Degradation", "Severity",
        "Precision_Before", "Precision_Blanket",
        "Recall_Before", "Recall_Blanket",
        "mAP50_Before", "mAP50_Blanket",
        "mAP50_95_Before", "mAP50_95_Blanket",
        "mAP50_Improvement_Pct"]
print(merged[cols].to_string(index=False))

Saved: /kaggle/working/phase5_blanket/results/before_vs_blanket_comparison.csv
   Degradation  Severity  Precision_Before  Precision_Blanket  Recall_Before  Recall_Blanket  mAP50_Before  mAP50_Blanket  mAP50_95_Before  mAP50_95_Blanket  mAP50_Improvement_Pct
         Clean         0          0.869361           0.866675       0.762934        0.748620      0.828040       0.822813         0.625484          0.617245              -0.631172
Gaussian Noise         1          0.855555           0.869838       0.716088        0.748908      0.783998       0.823093         0.590691          0.617208               4.986637
Gaussian Noise         2          0.775929           0.865875       0.576955        0.740295      0.636428       0.818875         0.484954          0.614602              28.667265
Gaussian Noise         3          0.783000           0.866930       0.465214        0.732350      0.516404       0.817555         0.402309          0.611899              58.316753
Gaussian Noise       

## Cell 12 — Save config + ZIP

In [12]:
config = {
    "run_type": "Blanket augmentation (ablation control, no evidence gate)",
    "before_model_path": BEFORE_MODEL_PATH,
    "blanket_model_path": bundled_blanket_path,
    "training_hyperparameters": {
        "epochs": TRAIN_EPOCHS, "batch": TRAIN_BATCH, "optimizer": TRAIN_OPTIMIZER,
        "lr0": TRAIN_LR0, "seed": TRAIN_SEED,
    },
    "blanket_augmentation": {
        "lowlight_prob": BLANKET_LOWLIGHT_PROB, "lowlight_range": BLANKET_LOWLIGHT_FACTOR_RANGE,
        "noise_prob": BLANKET_NOISE_PROB, "noise_range": BLANKET_NOISE_STD_RANGE,
        "blur_prob": BLANKET_BLUR_PROB, "blur_kernel_range": BLANKET_BLUR_KERNEL_RANGE,
    },
    "class_names": CLASS_NAMES,
}
config_path = os.path.join(OUTPUT_ROOT, "phase5_blanket_config.json")
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)
print("Saved:", config_path)

zip_path = shutil.make_archive("/kaggle/working/phase5_blanket_results", "zip", OUTPUT_ROOT)
print("\nFINAL ZIP:")
print(" ", zip_path)

Saved: /kaggle/working/phase5_blanket/phase5_blanket_config.json

FINAL ZIP:
  /kaggle/working/phase5_blanket_results.zip
